## Memory

Goal: Implement the MDNRNN component of the world model

In [30]:
import math

import torch
from torch import nn
from torch.nn import functional as F

from world_models.models.mdn_rnn import MDNHead, MDNRNN
from world_models.training.mdn_rnn import mdn_loss

In [31]:
model = MDNRNN()

z = torch.randn(2, 4, 32)        # Four observations
actions = torch.randn(2, 3, 3)  # Three transitions

logits, mu, log_std, state = model(z[:, :-1], actions)
loss = mdn_loss(logits, mu, log_std, z[:, 1:])

print("Mixture parameters:", logits.shape)  # [2, 3, 32, 5]
print("Final hidden state:", state[0].shape) # [1, 2, 256]
print("Final cell state:", state[1].shape)   # [1, 2, 256]
print("NLL:", loss.item())

Mixture parameters: torch.Size([2, 3, 32, 5])
Final hidden state: torch.Size([1, 2, 256])
Final cell state: torch.Size([1, 2, 256])
NLL: 42.39922332763672


In [32]:
model.eval()

with torch.no_grad():
    # Process the whole sequence.
    full_logits, full_mu, full_log_std, _ = model(
        z[:, :-1], actions
    )

    # Process one timestep at a time, carrying both LSTM states.
    state = None
    collected = []

    for t in range(actions.shape[1]):
        logits_t, mu_t, log_std_t, state = model(
            z[:, t:t+1],
            actions[:, t:t+1],
            state=state,
        )
        collected.append((logits_t, mu_t, log_std_t))

    stepwise = [
        torch.cat([outputs[i] for outputs in collected], dim=1)
        for i in range(3)
    ]

    for name, full, stepped in zip(
        ["logits", "mu", "log_std"],
        [full_logits, full_mu, full_log_std],
        stepwise,
    ):
        print(name, torch.allclose(
            full, stepped, atol=1e-6, rtol=1e-5
        ))

logits True
mu True
log_std True
